# 08 — Evaluation
**Dự án:** Latent Manipulation of Brain MRI using Volume-Preserving GANs

**Metrics đánh giá 4 model:**
- **SSIM** — độ giữ nguyên cấu trúc giải phẫu (cao hơn = tốt hơn)
- **Loss_G** — loss Generator tốt nhất (thấp hơn = tốt hơn)

**Output:**
```
evaluation/
└── metrics_summary.json
```

## Bước 1: Cấu hình

In [ ]:
import os
import json

# ==== CHECKPOINTS ====
MODEL_2D_NORM_PATH   = '/kaggle/input/gan2d-normalized/best_model.pth'
MODEL_2D_UNNORM_PATH = '/kaggle/input/gan2d-unnormalized/best_model.pth'
MODEL_3D_NORM_PATH   = '/kaggle/input/gan3d-normalized/best_model.pth'
MODEL_3D_UNNORM_PATH = '/kaggle/input/gan3d-unnormalized/best_model.pth'

# ==== DATA ====
DATA_2D_NORM_DIR   = '/kaggle/input/datasets/minhbodoi/dxhgbfhdhf/normalized'
DATA_2D_UNNORM_DIR = '/kaggle/input/datasets/minhbodoi/dxhgbfhdhf/unnormalized'
DATA_3D_NORM_DIR   = '/kaggle/input/dghgyfgfh/normalized'
DATA_3D_UNNORM_DIR = '/kaggle/input/dghgyfgfh/unnormalized'
LABELS_2D_CSV      = '/kaggle/input/datasets/minhbodoi/dxhgbfhdhf/preprocessing_log.csv'
LABELS_3D_CSV      = '/kaggle/input/dghgyfgfh/preprocessing_log.csv'

OUTPUT_DIR = '/kaggle/working/evaluation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

LATENT_DIM = 256
print('Cấu hình xong!')

## Bước 2: Import thư viện

In [ ]:
!pip install nibabel scikit-image -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import nibabel as nib
import numpy as np
import pandas as pd
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## Bước 3: Định nghĩa kiến trúc model

In [ ]:
class AgeEmbedding(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 128), nn.ReLU(), nn.Linear(128, embed_dim))
    def forward(self, age): return self.net(age.unsqueeze(-1))


class UNetBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = []
        if down : layers.append(nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False))
        else    : layers.append(nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False))
        if use_bn  : layers.append(nn.BatchNorm2d(out_ch))
        if dropout : layers.append(nn.Dropout(0.5))
        layers.append(nn.LeakyReLU(0.2) if down else nn.ReLU())
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class Generator2D(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.age_embed = AgeEmbedding(latent_dim)
        self.age_proj  = nn.Linear(latent_dim, 512)
        self.e1=UNetBlock2D(1,64,down=True,use_bn=False); self.e2=UNetBlock2D(64,128,down=True)
        self.e3=UNetBlock2D(128,256,down=True);           self.e4=UNetBlock2D(256,512,down=True)
        self.e5=UNetBlock2D(512,512,down=True);           self.e6=UNetBlock2D(512,512,down=True)
        self.e7=UNetBlock2D(512,512,down=True);           self.e8=UNetBlock2D(512,512,down=True,use_bn=False)
        self.d1=UNetBlock2D(512,512,down=False,dropout=True);  self.d2=UNetBlock2D(1024,512,down=False,dropout=True)
        self.d3=UNetBlock2D(1024,512,down=False,dropout=True); self.d4=UNetBlock2D(1024,512,down=False)
        self.d5=UNetBlock2D(1024,256,down=False);              self.d6=UNetBlock2D(512,128,down=False)
        self.d7=UNetBlock2D(256,64,down=False)
        self.out=nn.Sequential(nn.ConvTranspose2d(128,1,4,2,1),nn.Tanh())
    def forward(self, x, age):
        e1=self.e1(x);e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        e5=self.e5(e4);e6=self.e6(e5);e7=self.e7(e6);e8=self.e8(e7)
        z=e8+self.age_proj(self.age_embed(age)).view(-1,512,1,1)
        d1=self.d1(z);d2=self.d2(torch.cat([d1,e7],1));d3=self.d3(torch.cat([d2,e6],1))
        d4=self.d4(torch.cat([d3,e5],1));d5=self.d5(torch.cat([d4,e4],1))
        d6=self.d6(torch.cat([d5,e3],1));d7=self.d7(torch.cat([d6,e2],1))
        return self.out(torch.cat([d7,e1],1))


class UNetBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, down=True, use_bn=True, dropout=False):
        super().__init__()
        layers = []
        if down : layers.append(nn.Conv3d(in_ch, out_ch, 4, 2, 1, bias=False))
        else    : layers.append(nn.ConvTranspose3d(in_ch, out_ch, 4, 2, 1, bias=False))
        if use_bn  : layers.append(nn.BatchNorm3d(out_ch))
        if dropout : layers.append(nn.Dropout(0.5))
        layers.append(nn.LeakyReLU(0.2) if down else nn.ReLU())
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class Generator3D(nn.Module):
    def __init__(self, latent_dim=256):
        super().__init__()
        self.age_embed=AgeEmbedding(latent_dim); self.age_proj=nn.Linear(latent_dim,256)
        self.e1=UNetBlock3D(1,32,down=True,use_bn=False); self.e2=UNetBlock3D(32,64,down=True)
        self.e3=UNetBlock3D(64,128,down=True);             self.e4=UNetBlock3D(128,256,down=True,use_bn=False)
        self.d1=UNetBlock3D(256,128,down=False,dropout=True); self.d2=UNetBlock3D(256,64,down=False)
        self.d3=UNetBlock3D(128,32,down=False)
        self.out=nn.Sequential(nn.ConvTranspose3d(64,1,4,2,1),nn.Tanh())
    def forward(self, x, age):
        e1=self.e1(x);e2=self.e2(e1);e3=self.e3(e2);e4=self.e4(e3)
        z=e4+self.age_proj(self.age_embed(age)).view(-1,256,1,1,1)
        d1=self.d1(z);d2=self.d2(torch.cat([d1,e3],1));d3=self.d3(torch.cat([d2,e2],1))
        return self.out(torch.cat([d3,e1],1))


def load_model_2d(path):
    ckpt=torch.load(path,map_location=DEVICE); G=Generator2D(LATENT_DIM).to(DEVICE)
    G.load_state_dict(ckpt['G_state']); G.eval(); return G, ckpt

def load_model_3d(path):
    ckpt=torch.load(path,map_location=DEVICE); G=Generator3D(LATENT_DIM).to(DEVICE)
    G.load_state_dict(ckpt['G_state']); G.eval(); return G, ckpt


G2D_norm,ck2n=load_model_2d(MODEL_2D_NORM_PATH);   G2D_unnorm,ck2u=load_model_2d(MODEL_2D_UNNORM_PATH)
G3D_norm,ck3n=load_model_3d(MODEL_3D_NORM_PATH);   G3D_unnorm,ck3u=load_model_3d(MODEL_3D_UNNORM_PATH)
print('Load model xong!')

## Bước 4: Dataset

In [ ]:
class Dataset2D(Dataset):
    def __init__(self, data_dir, labels_csv, age_min, age_max):
        self.age_min=age_min; self.age_max=age_max
        df=pd.read_csv(labels_csv)
        df['full_path']=df['png_file'].apply(lambda x: os.path.join(data_dir,x))
        self.df=df[df['full_path'].apply(os.path.exists)].reset_index(drop=True)
        self.transform=transforms.Compose([transforms.Resize((256,256)),transforms.ToTensor(),transforms.Normalize([0.5],[0.5])])
    def normalize_age(self,age): return 2*(age-self.age_min)/(self.age_max-self.age_min)-1
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; img=self.transform(Image.open(row['full_path']).convert('L'))
        return img, torch.tensor(self.normalize_age(row['age']),dtype=torch.float32)


def find_nii(data_dir, subject_id):
    for ext in ['.nii.gz','.nii']:
        path=os.path.join(data_dir,f'{subject_id}{ext}')
        if os.path.exists(path): return path
    return None


class Dataset3D(Dataset):
    def __init__(self, data_dir, labels_csv, age_min, age_max, volume_size):
        self.age_min=age_min; self.age_max=age_max; self.volume_size=volume_size
        df=pd.read_csv(labels_csv)
        df['nii_path']=df['subject_id'].apply(lambda x: find_nii(data_dir,x))
        self.df=df[df['nii_path'].notna()].reset_index(drop=True)
    def normalize_age(self,age): return 2*(age-self.age_min)/(self.age_max-self.age_min)-1
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; data=nib.load(row['nii_path']).get_fdata().astype(np.float32)
        vol=torch.tensor(data).unsqueeze(0).unsqueeze(0)
        vol=F.interpolate(vol,size=(self.volume_size,)*3,mode='trilinear',align_corners=False)
        return vol.squeeze(0)*2-1, torch.tensor(self.normalize_age(row['age']),dtype=torch.float32)


loader_2d_norm  =DataLoader(Dataset2D(DATA_2D_NORM_DIR,  LABELS_2D_CSV,ck2n['age_min'],ck2n['age_max']),batch_size=8,shuffle=False)
loader_2d_unnorm=DataLoader(Dataset2D(DATA_2D_UNNORM_DIR,LABELS_2D_CSV,ck2u['age_min'],ck2u['age_max']),batch_size=8,shuffle=False)
loader_3d_norm  =DataLoader(Dataset3D(DATA_3D_NORM_DIR,  LABELS_3D_CSV,ck3n['age_min'],ck3n['age_max'],ck3n['volume_size']),batch_size=1,shuffle=False)
loader_3d_unnorm=DataLoader(Dataset3D(DATA_3D_UNNORM_DIR,LABELS_3D_CSV,ck3u['age_min'],ck3u['age_max'],ck3u['volume_size']),batch_size=1,shuffle=False)
print('Dataset sẵn sàng!')

## Bước 5: Tính SSIM và tổng kết

In [ ]:
def eval_2d(G, loader, device):
    scores=[]; G.eval()
    with torch.no_grad():
        for imgs, ages in tqdm(loader, leave=False):
            imgs=imgs.to(device); ages=ages.to(device)
            fakes=G(imgs,ages)
            for i in range(imgs.size(0)):
                r=(imgs[i,0].cpu().numpy()+1)/2; f=(fakes[i,0].cpu().numpy()+1)/2
                scores.append(ssim(r,f,data_range=1.0))
    return float(np.mean(scores))


def eval_3d(G, loader, device):
    scores=[]; G.eval()
    with torch.no_grad():
        for vols, ages in tqdm(loader, leave=False):
            vols=vols.to(device); ages=ages.to(device)
            fakes=G(vols,ages)
            r=(vols[0,0].cpu().numpy()+1)/2; f=(fakes[0,0].cpu().numpy()+1)/2
            scores.append(float(np.mean([ssim(r[i],f[i],data_range=1.0) for i in range(r.shape[0])])))
    return float(np.mean(scores))


print('Đang tính SSIM...')
results = {
    'GAN_2D_normalized'  : {'best_loss_G': ck2n['best_loss_G'], 'ssim': eval_2d(G2D_norm,   loader_2d_norm,   DEVICE), 'best_epoch': ck2n['epoch']},
    'GAN_2D_unnormalized': {'best_loss_G': ck2u['best_loss_G'], 'ssim': eval_2d(G2D_unnorm, loader_2d_unnorm, DEVICE), 'best_epoch': ck2u['epoch']},
    'GAN_3D_normalized'  : {'best_loss_G': ck3n['best_loss_G'], 'ssim': eval_3d(G3D_norm,   loader_3d_norm,   DEVICE), 'best_epoch': ck3n['epoch']},
    'GAN_3D_unnormalized': {'best_loss_G': ck3u['best_loss_G'], 'ssim': eval_3d(G3D_unnorm, loader_3d_unnorm, DEVICE), 'best_epoch': ck3u['epoch']},
}

# In bảng tổng kết
print('\n' + '='*65)
print('EVALUATION SUMMARY')
print('='*65)
print(f'{"Model":<25} {"Best Epoch":>10} {"Loss_G":>10} {"SSIM":>10}')
print('-'*65)
for name, m in results.items():
    print(f'{name:<25} {m["best_epoch"]:>10} {m["best_loss_G"]:>10.4f} {m["ssim"]:>10.4f}')
print('='*65)

# Lưu JSON
with open(f'{OUTPUT_DIR}/metrics_summary.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nKết quả lưu tại: {OUTPUT_DIR}/metrics_summary.json')